# NLP Text Processing methods




### Bag of Words

Many machine learning methods cannot use strings as features, we have to encode it using numbers.

We can easily do this using __Bag Of Words (BOW)__ technique and marvelous __sklearn__ library:

In [1]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]

X = vectorizer.fit_transform(corpus)
print(X.toarray())

[[0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0]
 [1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1]
 [0 0 0 2 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 2 1 0 0 1 0 1 0 0]]


Each document is represented by the row. Values ranging from 0 to N represent whether and how many times the word occured in the document. You can see what word corresponds to which column by issuing get_feature_names() on vectorizer object.

In [2]:
vectorizer.get_feature_names()

AttributeError: 'CountVectorizer' object has no attribute 'get_feature_names'

This approach allows us to easily describe the whole corpora, but it lacks informations crucial for solving some tasks.

### TF-IDF

Look at word is. It is used in most documents many times, yet it does not tell us anything about them. Let's think about sentiment analysis: if words like great or awesome occur frequently in comparison with another documents it may suggest positive attitude.

TF-IDF is one way to encode this information and I'll walk you through it step by step.

First part of TF-IDF is, yes, you guessed it, TF, which means Term Frequency. It can be calculated as:

\begin{equation}
tf_{ij}=\frac{n_{ij}}{\sum n_{ij}},
\end{equation}
where $n_{ij}$ is the number of occurence of word $i$ in document $j$.

In [3]:
import numpy as np

corpus = [
'Tom has cat',
'Tom has fish',
'Tom is polish',
]

def tf(corpus):
    vec = CountVectorizer()
    bow_representation = vec.fit_transform(corpus)
    words_per_corpus = bow_representation.sum(axis=1)
    return np.divide(np.array(bow_representation.toarray()),np.array(words_per_corpus).reshape((5,))[:,None])


In [4]:
bow_representation=CountVectorizer().fit_transform(corpus)
bow_representation.toarray()

array([[1, 0, 1, 0, 0, 1],
       [0, 1, 1, 0, 0, 1],
       [0, 0, 0, 1, 1, 1]])

For each document we count how many times it occurred (BoW implementaion) and divide by the count of all words in this document.

Next part is IDF, which stands for Inverse Document Frequency:

\begin{equation}
idf=\log(\frac{N}{df_{t}}),
\end{equation}
where $N$ is the total number of documents and $df_{t}$ is number of documents containing $t$.

In [5]:
def idf(corpus):
    document_count = len(corpus)
    bow_representation = CountVectorizer().fit_transform(corpus)
    return np.log(document_count / np.count_nonzero(bow_representation.toarray(), axis=0))

First we calculate number of documents in corpus (number of rows in our case). Next, for each word, we calculate documents containing said word at least once.

Taking logarithm allows us to dampen the effect of idf. For example, the difference between term occuring in 40 out of 50 documents and 45 out of 50 documents will be smaller than difference between 1/50 and 5/50. This puts a bigger emphasis on rarely occuring terms as they are more informative.

Finally, for the whole thing to work, we simply multiply both:

In [6]:
def tf_idf(corpus):
    return tf(corpus) * idf(corpus)

Let's calculate it:

In [7]:
corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]

tfidf_result = tf_idf(corpus)

print(tfidf_result.shape)

(5, 30)


In Jupyter it's easier to display results with pandas:

In [8]:
import pandas as pd
pd.DataFrame(tfidf_result).head()

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.000000,0.22992,0.22992,0.000000,0.000000,0.22992,0.000000,0.000000,0.000000,0.130899,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.072975,0.000000
1,0.000000,0.00000,0.00000,0.000000,0.000000,0.00000,0.000000,0.321888,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.321888,0.000000,0.102165,0.000000
2,0.000000,0.00000,0.00000,0.000000,0.000000,0.00000,0.321888,0.000000,0.000000,0.183258,...,0.000000,0.000000,0.000000,0.183258,0.321888,0.183258,0.000000,0.000000,0.000000,0.000000
3,0.146313,0.00000,0.00000,0.000000,0.146313,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.083299,0.000000,0.083299,0.000000,0.000000,0.000000,0.000000,0.046439,0.146313
4,0.000000,0.00000,0.00000,0.292625,0.000000,0.00000,0.000000,0.000000,0.146313,0.000000,...,0.146313,0.166598,0.146313,0.000000,0.000000,0.083299,0.000000,0.146313,0.000000,0.000000


There are many versions of tf-idf, some use different smoothing, use additional logarithm for tf part and so on. Each transforms corpora a little differently, and appropriate should be used based on effect we would like to obtain.

### Summary
This constitutes the first NLP-related part of our course and we have gathered some useful and interesting informations 

### References

[1] Natural Language Processing with Python, Edward Loper, Ewan Klein, Steven Bird. O'Reilly 2009

[2] Applied Text Analysis with Python, Tony Ojeda , Rebecca Bilbro , Benjamin Bengfort. O'Reilly 2018

[3] Feature Engineering for Machine Learning, Amanda Casari , Alice Zheng. O'Reilly 2018